# 第六课｜什么是 RTL？

现在需要把“有哪些端口、保存什么状态、每个 clock edge 怎么更新”明确写下来。今天只解决：
> **怎样用代码一样的文字描述数字硬件，而不是 CPU 顺序执行的指令？**

主要新概念：**寄存器传输级（Register-Transfer Level, RTL）**。


## 1. 四个词

**硬件描述语言（Hardware Description Language, HDL）**：描述数字硬件结构和行为的语言类别。

**SystemVerilog**：本项目使用的 HDL 与验证语言。

**RTL**：关注寄存器保存什么，以及数据如何跨 clock cycle 计算/传递。

**module / port**：module 是有边界的硬件单元；port 是跨边界的输入输出信号。


## 2. HDL 看起来像程序，但含义不同

Python 相邻两行通常表达先后执行；RTL 中很多描述表达**同时存在的硬件关系**。读 RTL 先问输入、输出、state、clock edge，而不是“第几行先执行”。


## 3. 一个最小 RTL：clocked accumulator

```systemverilog
module clocked_accumulator #(
    parameter int WIDTH = 8
) (
    input  logic clk,
    input  logic rst_n,
    input  logic signed [WIDTH-1:0] input_value,
    output logic signed [WIDTH-1:0] state
);
    always_ff @(posedge clk) begin
        if (!rst_n) state <= '0;
        else        state <= state + input_value;
    end
endmodule
```


## 4. 逐行读

`input/output` 是端口方向；`logic` 是常用信号类型；`[WIDTH-1:0]` 是位宽。

`always_ff @(posedge clk)` 描述在上升沿更新的寄存器。`<=` 是时序 RTL 常用的 **nonblocking assignment（非阻塞赋值）**。


## 5. reset 也是 contract

`rst_n` 末尾 `_n` 表示 active-low。这里是同步 reset：只有 clock edge 到来时 `rst_n=0` 才把 state 置零。reset polarity 和 timing 都必须明确。


## 6. Run

下面检测开源 simulator **Icarus Verilog**。它是系统工具，不由 `uv` 安装；如果没有，cell 明确显示“未运行”。


In [ ]:
from pathlib import Path
import shutil, subprocess, tempfile

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists(): return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root = repo_root(); iverilog = shutil.which('iverilog'); vvp = shutil.which('vvp')
if not (iverilog and vvp):
    print('Icarus Verilog not found; simulation did not run.')
else:
    with tempfile.TemporaryDirectory() as td:
        out = Path(td)/'l6.out'
        subprocess.run([iverilog,'-g2012','-o',str(out),str(root/'rtl/learning/clocked_accumulator.sv'),str(root/'tb/learning/clocked_accumulator_tb.sv')],check=True)
        r=subprocess.run([vvp,str(out)],check=True,text=True,capture_output=True)
        print(r.stdout.strip())


## 7. Observe / Try It

testbench 输入 `1,2,3`，检查 edge 后 state 为 `1,3,6`，通过时打印 `PASS lesson06 clocked_accumulator`。

先手算 `[2,-1,4]`，再自行给 testbench 增加这组检查。


## 8. AI Task / Human Check

让 AI 只标注现有 RTL 的 module、parameter、ports、state、clocked update，不改接口。

不用 AI 应能解释 HDL 与普通软件的语义差别、module/port、state 在哪里、`posedge` 表示什么、reset 为什么是 contract。


## 9. Engineering Handoff / Project Trace

`rtl/learning/clocked_accumulator.sv` 与 testbench 是教学 artifact，不是正式 `MOD-003`。

- Lesson: `LSN-006`
- Prepares: `RMD-004`
- RTL: `rtl/learning/clocked_accumulator.sv`
- Check: `tb/learning/clocked_accumulator_tb.sv`


## 10. Exit Ticket

你能解释 RTL、HDL、SystemVerilog、module、port，并从小模块指出输入、输出、state 与 clocked update。
